# EPITA — Spectacular Rhetorical Analysis Tour

**Interactive companion notebook** for the Spectacular Rhetorical Analysis system.

This notebook walks through all 6 capabilities of the analysis pipeline in increasing complexity:

1. **Extraction & Claims** — Raw text to structured facts
2. **Formal Logic** — NL to FOL/Modal/Propositional
3. **Fallacy Detection** — 8-family taxonomy
4. **Argumentation Frameworks** — Dung extensions + JTMS beliefs
5. **Adversarial Debate** — Multi-agent counter-arguments
6. **Unified Synthesis** — 32-field comprehensive analysis

> Each cell is independently executable. Mock data is used (no API keys required).

## Setup

In [ ]:
import json
import sys
from pathlib import Path

project_root = Path("..").resolve()
sys.path.insert(0, str(project_root / "examples" / "02_core_system_demos" / "scripts_demonstration"))

from demonstration_epita_spectacular import MOCK_SPECTACULAR_RESULT, MOCK_SOURCE, STEP_KEYS

print(f"Project root: {project_root}")
print(f"Source text ({len(MOCK_SOURCE)} chars): {MOCK_SOURCE[:100]}...")
print(f"Steps available: {len(STEP_KEYS)}")
print("\nSetup complete!")

## Step 1: Extraction & Claims

The pipeline isolates **verifiable claims** from raw discourse. Each claim is scored for factual grounding potential.

In [ ]:
step1 = MOCK_SPECTACULAR_RESULT["steps"]["1_extraction"]["output"]

print(f"Source: {MOCK_SPECTACULAR_RESULT['source_id']} — {MOCK_SPECTACULAR_RESULT['source_title']}")
print(f"\nExtracted {len(step1['extracted_claims'])} claims in {step1['extraction_stats']['time_ms']}ms:\n")

for claim in step1["extracted_claims"]:
    bar = "█" * int(claim["confidence"] * 20) + "░" * (20 - int(claim["confidence"] * 20))
    print(f"  [{bar}] {claim['confidence']:.0%}\n  {claim['text']}\n  type={claim['type']}\n")

print(f"Arguments reconstructed: {len(step1['extracted_arguments'])}")
for arg in step1["extracted_arguments"]:
    premises = ", ".join(arg["premises"])
    print(f"  {premises} → {arg['conclusion']} ({arg['type']})")

## Step 2: Formal Logic Translation

Claims are translated into **First-Order Logic (FOL)**, **Propositional Logic**, and **Modal Logic** for automated reasoning via Tweety.

In [ ]:
step2 = MOCK_SPECTACULAR_RESULT["steps"]["2_formal_logic"]["output"]

print("First-Order Logic Translations:\n")
for f in step2["fol_formulas"]:
    status = "✓ VALID" if f["valid"] else "✗ INVALID"
    print(f"  {status}")
    print(f"    Natural: {f['natural']}")
    print(f"    Formal:  {f['formal']}")
    print()

print("Propositional Logic:")
for p in step2["propositional"]:
    tags = []
    if p["satisfiable"]: tags.append("SAT")
    if p["tautology"]: tags.append("TAUT")
    print(f"  [{', '.join(tags)}] {p['formula']}")

print(f"\nModal ({len(step2['modal'])} formula(s)):")
for m in step2["modal"]:
    print(f"  {m['formula']} ({m['system']}: {m['meaning']})")

vs = step2["validation_summary"]
print(f"\nValidation: {vs['valid']} valid, {vs['invalid']} invalid, {vs['tautology']} tautology out of {vs['total']} total")

## Step 3: Fallacy Taxonomy Detection

A **3-tier hybrid detector** (neural + symbolic + rule-based) classifies fallacies into 8 families with severity and confidence.

In [ ]:
step3 = MOCK_SPECTACULAR_RESULT["steps"]["3_fallacy_detection"]["output"]

severity_icons = {"high": "🔴", "medium": "🟡", "low": "🟢"}

print(f"Detected {len(step3['detected_fallacies'])} fallacies (avg confidence: {step3['confidence_avg']:.0%}):\n")

for fallacy in step3["detected_fallacies"]:
    icon = severity_icons.get(fallacy["severity"], "⚪")
    print(f"  {icon} [{fallacy['severity'].upper():>6s}] {fallacy['type']} → {fallacy['sub']}")
    print(f"       Excerpt: \"{fallacy['excerpt'][:55]}\"")
    print(f"       Why: {fallacy['explanation'][:60]}")
    print()

print("Family breakdown:")
for family, count in step3["family_counts"].items():
    bar = "█" * count + "░" * (3 - count) if count > 0 else "░" * 3
    print(f"  {family:>25s} {bar} {count}")

ss = step3["severity_summary"]
print(f"\nSeverity: {ss['high']} high / {ss['medium']} medium / {ss['low']} low")

## Step 4: Argumentation Frameworks (Dung + JTMS)

**Dung's abstract argumentation** computes which arguments are collectively acceptable. The **JTMS** tracks belief propagation with retraction cascades.

In [ ]:
step4 = MOCK_SPECTACULAR_RESULT["steps"]["4_argumentation_frameworks"]["output"]
dung = step4["dung"]
jtms = step4["jtms"]

print("=== Dung Argumentation Framework ===\n")
print("Arguments:")
for aid, content in dung["arguments"].items():
    print(f"  {aid}: {content}")

print(f"\nAttack graph ({len(dung['attacks'])} attacks):")
for atk in dung["attacks"]:
    print(f"  {atk['from']} --[{atk['type']}]--> {atk['to']}")

print(f"\nExtensions:")
print(f"  Grounded:  {{{', '.join(dung['extensions']['grounded'])}}}")
for i, pref in enumerate(dung["extensions"]["preferred"]):
    print(f"  Preferred {i+1}: {{{', '.join(pref)}}}")
if dung["extensions"]["stable"]:
    print(f"  Stable:    {{{', '.join(dung['extensions']['stable'][0])}}}")

print(f"\nAnalysis: {dung['analysis']}")

In [ ]:
print("=== JTMS Belief Network ===\n")

status_icons = {"IN": "✅", "OUT": "❌", "UNDECIDED": "❓"}

for bid, bdata in jtms["beliefs"].items():
    icon = status_icons.get(bdata["status"], "?")
    conf_bar = "█" * int(bdata["confidence"] * 10) + "░" * (10 - int(bdata["confidence"] * 10))
    line = f"  {icon} {bdata['status']:>10s} [{conf_bar}] {bid}"
    if "retracted_by" in bdata:
        line += f"  ← retracted by {bdata['retracted_by']}"
    print(line)

print(f"\nRetraction cascade:")
for step in jtms["retraction_cascade"]:
    print(f"  → {step}")

s = jtms["summary"]
print(f"\nSummary: {s['in']} IN / {s['out']} OUT / {s['undecided']} UNDECIDED")

## Step 5: Multi-Agent Adversarial Debate

Two agents debate — one **defends** the discourse, the other **attacks** it. A **governance module** votes on the final position.

In [ ]:
step5 = MOCK_SPECTACULAR_RESULT["steps"]["5_adversarial_debate"]["output"]

print("Debate Rounds:\n")
for r in step5["rounds"]:
    d_bar = "█" * int(r["score_defender"] * 10)
    a_bar = "█" * int(r["score_attacker"] * 10)
    print(f"  Round {r['round']}:")
    print(f"    🛡️  Defender: {r['defender'][:60]}")
    print(f"    ⚔️  Attacker: {r['attacker'][:60]}")
    print(f"    Score: [{d_bar}] {r['score_defender']:.1f} vs [{a_bar}] {r['score_attacker']:.1f}")
    print()

print("Counter-arguments generated:\n")
for ca in step5["counter_arguments"]:
    print(f"  [{ca['strategy']}] {ca['text'][:55]}")
    print(f"    Strength: {ca['strength']:.0%}\n")

gov = step5["governance"]
print(f"Governance vote: {gov['consensus']} (consensus strength: {gov['consensus_strength']:.0%})")
for method, result in gov["methods"].items():
    print(f"  {method:>12s}: {result}")

scores = step5["final_score"]
print(f"\n🏆 Winner: {step5['debate_winner'].upper()}")
print(f"   Final: defender {scores['defender']:.2f} vs attacker {scores['attacker']:.2f}")

## Step 6: Unified Synthesis

All signals converge into a **32-field comprehensive analysis** across 13 dimensions.

In [ ]:
step6 = MOCK_SPECTACULAR_RESULT["steps"]["6_synthesis"]["output"]

print(f"Overall Assessment: {step6['overall_assessment']}\n")

qs = step6["quality_score"]
print("Quality Dimensions:")
for dim in ("clarity", "coherence", "relevance", "evidence", "logic", "completeness"):
    val = qs[dim]
    bar = "█" * int(val * 20) + "░" * (20 - int(val * 20))
    print(f"  {dim:>13s} {bar} {val:.0%}")
print(f"  {'OVERALL':>13s} {'█' * int(qs['overall'] * 20) + '░' * (20 - int(qs['overall'] * 20))} {qs['overall']:.0%}")

print(f"\nFallacy density: {step6['fallacy_density']}")
print(f"Strongest counter: {step6['strongest_counter']}")
print(f"\nRecommendation:\n{step6['recommendation']}")

print(f"\n{step6['field_count']} fields across {len(step6['dimensions'])} dimensions:")
for i, dim in enumerate(step6["dimensions"], 1):
    print(f"  {i:2d}. {dim}")

## Raw Data Export

The complete analysis result as JSON — suitable for further processing or visualization.

In [ ]:
print(json.dumps(MOCK_SPECTACULAR_RESULT, indent=2, ensure_ascii=False)[:2000])
print("\n... (truncated, full output available via json.dumps)")